# 交叉验证：同一机制在 vLLM 上也成立吗

## 为什么要做这一轮

SGLang 那边定位到：解码 CUDA graph 只捕获 `bs=[1,2,4,8]`，
batch > 8 回落 eager，TPOT 一步跳 4.8 倍。

但这可能是 **SGLang 的实现问题**，也可能是 **「图捕获分桶」这个通用机制**。
分辨方法：vLLM 也用 CUDA graph 分桶，且它自己的捕获范围与 SGLang 不同。

## 跑前写死的预测

| 若…… | 则说明…… |
|---|---|
| vLLM 的 TPOT 台阶出现在**它自己**的捕获上限之后 | 这是**通用机制**（图捕获边界），不是某个框架的 bug |
| vLLM 全程无台阶（捕获范围覆盖了整个测试区间） | 与机制一致，SGLang 的差别在于 T4 档位表把上限钉在 8 |
| vLLM 有台阶但位置与捕获范围对不上 | 机制解释不成立，要重查 |

**外加一个直接对照**：`--enforce-eager` 完全禁用 CUDA graph。
预测：TPOT 在所有 batch 上都升到「无图」水平且**没有台阶**。
这一条若不成立，机制解释直接被推翻。

**再加一个反向复现**：用 `--compilation-config '{"cudagraph_capture_sizes": [1, 2, 4, 8]}'`
把 vLLM 的捕获桶砍到和 SGLang 在 T4 上一样只到 8。
预测：vLLM 也会在 8 → 12 处出现同样的台阶。**这一条成立，才算把机制坐实到框架之外。**

## 关键：捕获范围要从日志读回来，不能假设

vLLM 的日志措辞与 SGLang 不同。第 4 节会把所有含 `graph` / `capture` /
`cudagraph_capture_sizes` 的行原样打出来，**先看清楚它到底捕获了哪些 batch**，
再谈台阶位置对不对得上。


## 0. 环境

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv"], capture_output=True, text=True).stdout
print(out)
print("必须也是 Tesla T4，才与 SGLang 那轮可比。")


## 1. 装 vLLM（不装 torchaudio）

In [ ]:
import importlib.metadata as md_, subprocess, sys

CHECK = ["vllm", "aiohttp", "torchvision"]

def ver(p):
    try:
        return md_.version(p)
    except Exception:
        return None

def sh(c):
    return subprocess.run(c, capture_output=True, text=True)

missing = [p for p in CHECK if ver(p) is None]
print("缺失:", missing or "无")
if missing:
    print("装 vllm + aiohttp + torchvision（约 5-10 分钟）...")
    r = sh([sys.executable, "-m", "pip", "install", "-q", "vllm", "aiohttp", "torchvision"])
    print("退出码:", r.returncode)
    if r.returncode:
        print(r.stdout[-2500:]); print(r.stderr[-2500:])
else:
    print("三个包都在，跳过安装。")
u = sh([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"])
print("卸 torchaudio 退出码:", u.returncode)
print()
for p in CHECK + ["transformers", "torch"]:
    print("  %-14s %s" % (p, ver(p) or "（未安装）"))


## 2. 探测参数

`--enforce-eager` 与图相关参数。**对不上就停。**


In [ ]:
import subprocess, sys, re

r = subprocess.run([sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--help"],
                   capture_output=True, text=True)
h = r.stdout + r.stderr
print("help returncode:", r.returncode, "| 长度:", len(h))
print()
for flag in ["--enforce-eager", "--gpu-memory-utilization", "--no-enable-log-requests",
             "--compilation-config"]:
    print("  %-28s %s" % (flag, "存在" if flag in h else "不存在 !!"))
print()
print("图相关参数:")
for g in sorted(set(re.findall(r"--[a-z0-9-]*(?:cuda-graph|cudagraph|compil)[a-z0-9-]*", h))):
    print("   ", g)


## 3. 写探针（与 SGLang 那轮同一形状）

In [ ]:
import io

P = []
P.append('# -*- coding: utf-8 -*-')
P.append('import argparse, asyncio, json, statistics as st, time')
P.append('import aiohttp')
P.append('URL = "http://127.0.0.1:8000/v1/chat/completions"')
P.append('MODEL = "Qwen/Qwen2.5-0.5B-Instruct"')
P.append('')
P.append('async def one(sess, prompt, max_tokens):')
P.append('    body = {"model": MODEL, "messages": [{"role":"user","content":prompt}],')
P.append('            "max_tokens": max_tokens, "temperature": 0.0, "stream": True}')
P.append('    t0 = time.perf_counter(); ttft=None; n=0; last=t0')
P.append('    async with sess.post(URL, json=body) as resp:')
P.append('        async for raw in resp.content:')
P.append('            line = raw.decode("utf-8").strip()')
P.append('            if not line.startswith("data: ") or line == "data: [DONE]": continue')
P.append('            d = json.loads(line[6:])["choices"][0].get("delta", {})')
P.append('            if d.get("content"):')
P.append('                now = time.perf_counter()')
P.append('                if ttft is None: ttft = now - t0')
P.append('                n += 1; last = now')
P.append('    return dict(ttft=ttft or 0.0, total=last-t0, n_tok=n)')
P.append('')
P.append('async def run_batch(n_conc, n_req, max_tokens):')
P.append('    prompts = ["Explain concept #%d in distributed systems." % i for i in range(n_req)]')
P.append('    sem = asyncio.Semaphore(n_conc)')
P.append('    async def guarded(sess, p):')
P.append('        async with sem: return await one(sess, p, max_tokens)')
P.append('    to = aiohttp.ClientTimeout(total=900)')
P.append('    async with aiohttp.ClientSession(timeout=to) as sess:')
P.append('        await one(sess, "warmup", 4)')
P.append('        t0 = time.perf_counter()')
P.append('        rs = await asyncio.gather(*(guarded(sess, p) for p in prompts))')
P.append('        wall = time.perf_counter() - t0')
P.append('    tot = sum(r["n_tok"] for r in rs)')
P.append('    tps = [(r["total"]-r["ttft"])/max(r["n_tok"]-1,1) for r in rs if r["n_tok"]>1]')
P.append('    return dict(conc=n_conc, wall=wall, tput=tot/wall,')
P.append('                ttft_p50=st.median(r["ttft"] for r in rs),')
P.append('                tpot_p50=st.median(tps) if tps else 0.0)')
P.append('')
P.append('async def main(a):')
P.append('    out = []')
P.append('    print("%6s %9s %12s %11s %11s" % ("batch","墙钟s","吞吐tok/s","TTFTp50","TPOTp50"))')
P.append('    print("-"*54)')
P.append('    for c in [int(x) for x in a.conc.split(",")]:')
P.append('        r = await run_batch(c, max(c*4, 16), a.max_tokens)')
P.append('        out.append(r)')
P.append('        print("%6d %9.2f %12.1f %10.1fms %10.2fms" % (')
P.append('              c, r["wall"], r["tput"], r["ttft_p50"]*1e3, r["tpot_p50"]*1e3))')
P.append('    json.dump(out, open(a.out, "w"))')
P.append('')
P.append('if __name__ == "__main__":')
P.append('    ap = argparse.ArgumentParser()')
P.append('    ap.add_argument("--conc", default="4,8,12,16,24,32")')
P.append('    ap.add_argument("--max-tokens", type=int, default=128)')
P.append('    ap.add_argument("--out", default="p.json")')
P.append('    asyncio.run(main(ap.parse_args()))')

io.open("probe.py", "w", encoding="utf-8").write(chr(10).join(P))
print("写出 probe.py，", len(P), "行")


## 4. 启动器 —— 把 vLLM 的图捕获信息原样挖出来

vLLM 的日志措辞与 SGLang 不同，先原样打出来看清楚，再谈对不对得上。


In [ ]:
import subprocess, sys, time, requests, re

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

def serve(extra, tag, wait=600):
    subprocess.run(["pkill", "-f", "vllm"], check=False)
    time.sleep(12)
    log = "/content/x_%s.log" % tag
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
           "--model", MODEL, "--host", "127.0.0.1", "--port", "8000",
           "--max-model-len", "2048", "--no-enable-log-requests"] + extra
    print("启动 [%s]:" % tag, " ".join(cmd[6:]))
    lg = open(log, "w")
    p = subprocess.Popen(cmd, stdout=lg, stderr=subprocess.STDOUT)
    for i in range(wait // 2):
        if p.poll() is not None:
            print("  退出码 %s：" % p.returncode)
            print(open(log).read()[-3000:])
            return None, ""
        try:
            if requests.get("http://127.0.0.1:8000/v1/models", timeout=2).status_code == 200:
                print("  就绪，用时 %ds" % (i * 2))
                txt = open(log).read()
                hits = [l.strip() for l in txt.splitlines()
                        if re.search(r"graph|capture|Capturing|eager", l, re.I)]
                print("  含 graph/capture/eager 的日志行（前 8 条）:")
                for l in hits[:8]:
                    print("    |", l[:170])
                return p, txt
        except requests.RequestException:
            pass
        time.sleep(2)
    print("  %ds 没起来：" % wait)
    print(open(log).read()[-3000:])
    return None, ""


def capture_sizes(txt):
    """从 vLLM 日志里尽量抠出捕获的 batch 尺寸集合。措辞不定，多种模式都试。"""
    if not txt:
        return None
    for pat in [r"cudagraph_capture_sizes[^\[]*\[([^\]]*)\]",
                r"capture_sizes[^\[]*\[([^\]]*)\]",
                r"Capturing CUDA graphs?[^\[]*\[([^\]]*)\]"]:
        m = re.search(pat, txt)
        if m:
            try:
                return sorted(int(x) for x in re.findall(r"\d+", m.group(1)))
            except (ValueError, TypeError):
                pass          # 只吞解析失败，换下一个模式；其它异常照抛
    return None


## 5. 三组：默认（有图）／ `--enforce-eager`（无图）／ 捕获桶砍到 8（反向复现）

In [ ]:
import subprocess, sys, json, os

RES, SIZES = {}, {}
for tag, extra in [("graph_on", ["--gpu-memory-utilization", "0.80"]),
                   ("eager",    ["--gpu-memory-utilization", "0.80", "--enforce-eager"]),
                   ("cap8",     ["--gpu-memory-utilization", "0.80", "--compilation-config",
                                 '{"cudagraph_capture_sizes": [1, 2, 4, 8]}'])]:
    print("=" * 62)
    p, txt = serve(extra, tag)
    SIZES[tag] = capture_sizes(txt)
    print("  解析出的捕获尺寸:", SIZES[tag] if SIZES[tag] else "（未解析到）")
    if not p:
        print("  起服务失败，跳过", tag)
        continue
    r = subprocess.run([sys.executable, "-u", "probe.py",
                        "--conc", "4,8,12,16,24,32", "--out", "p.json"],
                       capture_output=True, text=True)
    print(r.stdout)
    if os.path.exists("p.json"):
        RES[tag] = {row["conc"]: row for row in json.load(open("p.json"))}
        os.rename("p.json", "x_%s.json" % tag)
print("=" * 62)
print("完成:", list(RES.keys()))


## 6. 判定

In [ ]:
BS = [4, 8, 12, 16, 24, 32]

print("%-11s" % "TPOT p50 ms", "".join("%9d" % b for b in BS))
print("-" * (11 + 9 * len(BS)))
for tag in ["graph_on", "eager", "cap8"]:
    d = RES.get(tag, {})
    print("%-11s" % tag, "".join(
        ("%9.2f" % (d[b]["tpot_p50"] * 1e3)) if b in d else "%9s" % "-" for b in BS))

def step_at(tag):
    d = RES.get(tag, {})
    out = []
    for i in range(len(BS) - 1):
        a, b = BS[i], BS[i + 1]
        if a in d and b in d and d[a]["tpot_p50"] > 0:
            if d[b]["tpot_p50"] / d[a]["tpot_p50"] > 2.5:
                out.append((a, b))
    return out

print()
print("捕获尺寸: graph_on =", SIZES.get("graph_on"), "| eager =", SIZES.get("eager"), "| cap8 =", SIZES.get("cap8"))
print("台阶位置: graph_on =", step_at("graph_on") or "无", "| eager =", step_at("eager") or "无",
      "| cap8 =", step_at("cap8") or "无")
print()
print("=== 按跑前写死的预测判定 ===")
sg, se = step_at("graph_on"), step_at("eager")
dg, de = RES.get("graph_on", {}), RES.get("eager", {})

if se:
    print("• eager 组**出现了台阶** —— 与预测矛盾（无图就不该有图边界）。机制解释存疑。")
elif de:
    vals = [de[b]["tpot_p50"] * 1e3 for b in BS if b in de]
    print("• eager 组无台阶，TPOT %.2f ~ %.2f ms —— 与预测一致。" % (min(vals), max(vals)))

if not sg:
    sz = SIZES.get("graph_on")
    print("• graph_on 组也无台阶。若捕获尺寸覆盖到 32（实际 %s），" % sz)
    print("  说明 vLLM 的捕获范围足够大，**与机制一致**：SGLang 在 T4 上的差别只是上限被档位表钉在 8。")
else:
    sz = SIZES.get("graph_on")
    print("• graph_on 组台阶在", sg, "，捕获尺寸", sz)
    if sz and sg[0][0] >= max(sz):
        print("  台阶落在捕获上限之后，**通用机制成立**。")
    else:
        print("  台阶与捕获范围对不上，**机制解释不成立**，照实记。")

sc = step_at("cap8")
if RES.get("cap8"):
    if sc and sc[0] == (8, 12):
        print("• cap8 组台阶恰在 8 → 12，捕获尺寸", SIZES.get("cap8"),
              "—— **在 vLLM 上复现了 SGLang 的台阶，机制在框架之外也成立**。")
    elif sc:
        print("• cap8 组有台阶但在", sc, "，与捕获上限 8 对不上，照实记。")
    else:
        print("• cap8 组**没有台阶** —— 砍桶没有造成台阶：要么参数没生效（看捕获尺寸），")
        print("  要么机制解释被推翻。两种都照实记，不挑好听的。")
else:
    print("• cap8 组没有数据（起服务失败），反向复现这一条记为未测。")

if dg and de and 8 in dg and 8 in de:
    print()
    print("• batch 8：有图 %.2f ms vs 无图 %.2f ms，比值 %.2f×" % (
        dg[8]["tpot_p50"] * 1e3, de[8]["tpot_p50"] * 1e3,
        de[8]["tpot_p50"] / dg[8]["tpot_p50"]))
    print("  这个比值就是 CUDA graph 在该 batch 上的净收益。")


## 7. 附：核对一个写进结果文件的旧前提 —— T4 上 vLLM 0.28 到底有没有 Marlin

量化那轮的结果文件写着「T4 无 Marlin」。查 vLLM v0.28.0 源码
`marlin_utils.py::query_marlin_supported_quant_types` 的门槛是 `if device_capability < 75: return []`，
即 **sm_75 在支持范围内**（Turing 支持由 PR #29901 加入）。
这里不猜：起一次 GPTQ-Int4 与 AWQ 服务，把日志里含 marlin / kernel 的行原样打出来。
**日志说了算**。若确实走了 Marlin，量化那轮「预测错」的原因要改写：前提（无 Marlin）就是错的，不只是推论错。


In [ ]:
import subprocess, sys, time, requests, re

def probe_kernel(model, tag):
    subprocess.run(["pkill", "-f", "vllm"], check=False)
    time.sleep(12)
    log = "/content/k_%s.log" % tag
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
           "--model", model, "--host", "127.0.0.1", "--port", "8000",
           "--max-model-len", "2048", "--gpu-memory-utilization", "0.80",
           "--no-enable-log-requests"]
    lg = open(log, "w")
    p = subprocess.Popen(cmd, stdout=lg, stderr=subprocess.STDOUT)
    ok = False
    for i in range(300):
        if p.poll() is not None:
            break
        try:
            if requests.get("http://127.0.0.1:8000/v1/models", timeout=2).status_code == 200:
                ok = True
                break
        except requests.RequestException:
            pass
        time.sleep(2)
    txt = open(log).read()
    hits = [l.strip() for l in txt.splitlines() if re.search(r"marlin|kernel|quantization|Falling back", l, re.I)]
    print("[%s] 就绪=%s | 含 marlin/kernel/quantization 的日志行 %d 条（前 12 条）:" % (tag, ok, len(hits)))
    for l in hits[:12]:
        print("    |", l[:200])
    if not ok:
        print("    （服务没起来，日志尾部）")
        print(txt[-2000:])
    subprocess.run(["pkill", "-f", "vllm"], check=False)
    return hits

K = {}
for model, tag in [("Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4", "gptq"),
                   ("Qwen/Qwen2.5-1.5B-Instruct-AWQ", "awq")]:
    K[tag] = probe_kernel(model, tag)
    print()

for tag in K:
    has = any("marlin" in l.lower() for l in K[tag])
    print("%-5s 日志里%s marlin 字样 → %s" % (tag, "有" if has else "没有",
          "T4 上走了 Marlin，旧前提「T4 无 Marlin」不成立" if has else "未见 Marlin 字样，旧前提暂不能推翻，看上面原文"))


## 8. 边界

- vLLM 与 SGLang 的默认后端不同（vLLM 走自己的默认，SGLang 那轮走 Triton+PyTorch），
  **纵向比两框架绝对值无意义**；本轮比的是**各自内部「有图 vs 无图」的结构**。
- `capture_sizes()` 用多种正则去抠日志，**若返回 None 就说明没解析到**，
  这时不要凭印象说 vLLM 捕获了多少 —— 照实写「未从日志确认」。
- cap8 组的 `--compilation-config` 是否真的把桶砍到 8，以日志解析出的捕获尺寸为准；
  解析不到就写「未确认」。
- 每档单次测量；台阶判据是「相邻档跳幅 > 2.5 倍」的阈值判断，不是统计检验。
- 单卡 T4、0.5B、2048 上下文。
